# Basic example

**Index:**
1. [Intro](#intro)
2. [Featurisation](#featurisation)
3. [Training](#training)
4. [Applying](#applying)

## Intro
Trainable/interactive segmentation trains a classifier to map from features that describe a set of pixels to user-drawn class labels. The trained classifier can then be applied to the features of unlabelled pixels, predicting a class for each of them and producing a segmentation.

It is convenient for a few reasons:
- it allows flexible class definitions
- it requires a small number of sparse labels (compared to a CNN)
- it trains and segments quickly 
- it is iterative: new labels can be added to correct mistakes

Examples of existing interactive segmentation implementations are [Weka](https://imagej.net/plugins/tws/), [ilastik](https://www.ilastik.org/) and [napari-apoc](https://github.com/haesleinhuepf/napari-accelerated-pixel-and-object-classification). I've written this library as an ergonomic python interface that allows me to experiment with different parts of the workflow, such as [better post-processing](modal_filter_vs_CRF.ipynb), the addition of [vision transformer features](more_feature_functions.ipynb), and easy [GUI integration](https://github.com/tldr-group/interactive-seg-gui). 

There are three steps to interactive segmentation: 
1) featurisation, or the process of extracting features from the target image
2) training, i.e fitting a given classifier based on the features and user labels
3) applying the trained classifier to the unlabelled pixels

## Featurisation

The first step is to extract 'features' from our image. These features will be numeric values that describe elements or characteristics of a pixel and its neighbourhood. Typically these features are common computer vision filters, such as [Gaussian blurs](https://en.wikipedia.org/wiki/Gaussian_blur), edge detectors ([Sobel filter](https://en.wikipedia.org/wiki/Sobel_operator) etc.) and Hessian texture filters.


To capture image features on different length scales, a Gaussian blur with a given $\sigma$ can be applied before extracting the feature, i.e. a Gaussian blur of $\sigma = 4.0$ before applying a 3x3 Sobel filter. For an RGB image each channel can be featurised separately and stacked together.

Let's featurise an image using `isb`:

In [ ]:
from interactive_seg_backend.file_handling import load_image
from interactive_seg_backend import featurise, TrainingConfig, FeatureConfig

The first thing we do is define some configs that say which features we want to use, and how to train the model on them later. `isb` uses the default Weka features by default; for this example we disable Difference of Gaussians (DoG) and membrane projections.

In [ ]:
feature_config = FeatureConfig(
    gaussian_blur=True,
    sobel_filter=True,
    hessian_filter=True,
    difference_of_gaussians=False,
    membrane_projections=False,
    sigmas=(1, 2, 4, 8, 16))
training_config = TrainingConfig(feature_config=feature_config, classifier='random_forest')

In [ ]:
image = load_image('../tests/data/0.tif')
features = featurise(image, training_config) # (this just reads training_config.feature_config)
print(f"{image.shape} image -> {features.shape} features")
# We can see which channels correspond to which features by looking at the filter strings:
print(feature_config.get_filter_strings())

We can now visualise some of those features!

In [ ]:
import matplotlib.pyplot as plt

vis_indices = (1, 28, 29, 30)

fh, fw = 3, 3
fig, axs = plt.subplots(1, 1 + len(vis_indices), figsize=(fw * (1 + len(vis_indices)), fh))

img_ax = axs[0]
img_ax.imshow(image, cmap='gray')
img_ax.set_title('Image')
img_ax.axis('off')

for i, ind in enumerate(vis_indices):
    ax = axs[i + 1]
    ax.imshow(features[:, :, ind], cmap='gray')
    ax.set_title(feature_config.get_filter_strings()[ind])
    ax.axis('off')

## Training

Let's load some example labels that define the classes for our image. This is a Nickel superalloy from ['Microstructure segmentation with deep learning encoders pre-trained on a large microscopy dataset' by J. Stuckner et. al.](https://www.nature.com/articles/s41524-022-00878-5), and there's three classes: the large primary precipitate matrix, the blobby secondary precipitates and the tertiary precipitates (small inclusions in the matrix).

In [ ]:
from interactive_seg_backend.file_handling import load_labels
from interactive_seg_backend.utils import to_rgb_arr, apply_labels_as_overlay, PALETTE_RGB_NORM

labels = load_labels('../tests/data/0_labels.tif')

labels_overlaid = apply_labels_as_overlay(labels, to_rgb_arr(image), PALETTE_RGB_NORM[1:])
plt.figure(figsize=(fw, fh))
plt.imshow(labels_overlaid)
plt.gca().axis('off')


In [ ]:
import numpy as np

from interactive_seg_backend import get_model, get_training_data, train
print(f"There are {np.sum(labels > 0)} labelled pixels in the image, with {len(np.unique(labels)) - 1} classes.")

random_forest_args = {"n_estimators": 200, "max_depth": 20, "max_features": 2}
classifier = get_model("random_forest", random_forest_args)
# train_data and target_data are (N_labelled_px, N_features) and (N_labelled_px,) respectively
train_data, target_data = get_training_data([features], [labels])
classifier = train(classifier, train_data, target_data, None)

Whilst we're here, we can also try training a model without the third class, by setting all the labels for class 3 to that of class 1. Note that the features stay exactly the same, it's just our class definitions that are changing (via the labels).

In [ ]:
binary_labels = np.where(labels == 3, 1, labels)

binary_classifier = get_model("random_forest", random_forest_args)
binary_train_data, binary_target_data = get_training_data([features], [binary_labels])
binary_classifier = train(binary_classifier, binary_train_data, binary_target_data, None)

## Applying

In [ ]:
from interactive_seg_backend import apply
predicted_segmentation, _ = apply(classifier, features, training_config)
predicted_binary_segmentation, _ = apply(binary_classifier, features, training_config)

In [ ]:
from interactive_seg_backend.utils import recolor_labels

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])

N_ROWS, N_COLS = 2, 2
fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(fw * N_COLS, fh * N_ROWS))

labels_list, pred_list = [labels, binary_labels], [predicted_segmentation, predicted_binary_segmentation]
titles = ["Image + Labels", "Predicted Segmentation"]
y_lables = ["Multiclass", "Binary"]

for row_idx, row in enumerate(axs):
    for col_idx, ax in enumerate(row):
        hide_axis(ax)

        if row_idx == 0:
            ax.set_title(titles[col_idx], fontsize=12)
        if col_idx == 0:
            ax.set_ylabel(y_lables[row_idx], fontsize=12)
    img_with_labels = apply_labels_as_overlay(labels_list[row_idx], to_rgb_arr(image), PALETTE_RGB_NORM[1:])
    row[0].imshow(img_with_labels)
    row[1].imshow(recolor_labels(pred_list[row_idx] + 1))
plt.tight_layout()